In [8]:
import torch
torch.set_printoptions(precision=2)

# ── Config ──────────────────────────────────────────────────────
T = 5
d = 3
tokens = ["A", "B", "C", "D", "E"]

X = torch.tensor([
    [1, 0, 2],   # A
    [0, 1, 1],   # B
    [2, 1, 0],   # C
    [1, 2, 1],   # D
    [0, 0, 2],   # E
], dtype=torch.float)

Wq = torch.eye(d)
Wk = torch.tensor([[0,1,0],[1,0,0],[0,0,1]], dtype=torch.float)
Wv = torch.tensor([[1,0,0],[0,0,1],[0,1,0]], dtype=torch.float)

scale = d ** 0.5

def print_matrix(label, mat, row_labels=None, col_labels=None):
    print(f"\n  {label}  (shape {list(mat.shape)})")
    if row_labels is None:
        row_labels = [str(i) for i in range(mat.shape[0])]
    if col_labels is None:
        col_labels = [f"d{i}" for i in range(mat.shape[-1])]
    print(f"    {'':4} {col_labels}")
    for i, lbl in enumerate(row_labels):
        vals = [round(x, 2) for x in mat[i].tolist()] if mat.dim() > 1 else [round(mat[i].item(), 2)]
        print(f"    {lbl:4} {vals}")


# ════════════════════════════════════════════════════════════════
# PART 1:  FULL RECOMPUTE  (what happens during training)
# ════════════════════════════════════════════════════════════════
print("█" * 60)
print("  PART 1:  FULL RECOMPUTE  (no cache — training mode)")
print("█" * 60)
print("""
  All T tokens are processed at once.
  We compute Q, K, V for EVERY token,
  then do full (T × T) causal-masked attention.
""")

Q_full = X @ Wq    # (5, 3)
K_full = X @ Wk    # (5, 3)
V_full = X @ Wv    # (5, 3)

print_matrix("X  (input embeddings)", X, tokens)
print_matrix("Q = X @ Wq", Q_full, tokens)
print_matrix("K = X @ Wk", K_full, tokens)
print_matrix("V = X @ Wv", V_full, tokens)

scores_full = (Q_full @ K_full.T) / scale     # (5, 5)
causal_mask = torch.triu(torch.full((T, T), float('-inf')), diagonal=1)

print_matrix("Raw scores  (Q @ Kᵀ / √d)", scores_full, tokens, tokens)

print(f"\n  Causal mask:")
print(f"    {'':4} {tokens}")
for i, tok in enumerate(tokens):
    row = ["  ✓" if causal_mask[i,j] == 0 else "  ✗" for j in range(T)]
    print(f"    {tok}  {''.join(row)}")

scores_masked = scores_full + causal_mask
weights_full = torch.softmax(scores_masked, dim=-1)
out_full = weights_full @ V_full

print_matrix("Attention weights  (softmax of masked scores)", weights_full, tokens, tokens)
print_matrix("Output  (weights @ V)", out_full, tokens)

print(f"""
  ⚠️  The problem with full recompute during GENERATION:
  To generate token E, we recompute K and V for A, B, C, D
  even though they haven't changed since last step.
  That's {(T-1) * d} wasted multiply-adds just for K,V projections!
""")


# ════════════════════════════════════════════════════════════════
# PART 2:  WITH KV CACHE  (what happens during inference)
# ════════════════════════════════════════════════════════════════
print("\n" + "█" * 60)
print("  PART 2:  WITH KV CACHE  (autoregressive generation)")
print("█" * 60)
print("""
  Tokens arrive one at a time.
  Each step:
    1. Compute Q, K, V for the NEW token only
    2. APPEND new K, V to the cache
    3. Attend: new Q  ×  ALL cached K's
    4. Output: weights  ×  ALL cached V's

  K and V for old tokens are NEVER recomputed.
""")

# The cache starts empty
K_cache = torch.zeros(0, d)   # will grow from (0,3) → (1,3) → (2,3) → ...
V_cache = torch.zeros(0, d)

cached_outputs = []

for step in range(T):
    tok = tokens[step]
    x_new = X[step:step+1]    # (1, 3) — just this one token

    print(f"\n{'─' * 60}")
    print(f"  STEP {step}:  Token '{tok}' arrives")
    print(f"{'─' * 60}")

    # 1. Project ONLY the new token
    q_new = x_new @ Wq        # (1, 3)
    k_new = x_new @ Wk        # (1, 3)
    v_new = x_new @ Wv        # (1, 3)

    print(f"\n  New token embedding:  {x_new[0].tolist()}")
    print(f"  q_new = x @ Wq     :  {q_new[0].tolist()}")
    print(f"  k_new = x @ Wk     :  {k_new[0].tolist()}")
    print(f"  v_new = x @ Wv     :  {v_new[0].tolist()}")

    # 2. Append to cache
    K_cache = torch.cat([K_cache, k_new], dim=0)   # (step+1, 3)
    V_cache = torch.cat([V_cache, v_new], dim=0)   # (step+1, 3)

    cached_toks = tokens[:step+1]
    print(f"\n  KV Cache after append:")
    print_matrix(f"K_cache  (stores K for [{', '.join(cached_toks)}])",
                 K_cache, cached_toks)
    print_matrix(f"V_cache  (stores V for [{', '.join(cached_toks)}])",
                 V_cache, cached_toks)

    # 3. Attend: q_new (1, 3)  @  K_cache.T (3, step+1)  →  (1, step+1)
    scores = (q_new @ K_cache.T) / scale    # (1, step+1)

    print(f"\n  Scores = q_new @ K_cacheᵀ / √d")
    print(f"    {tok} attending to {cached_toks}:  {[round(x, 2) for x in scores[0].tolist()]}")
    print(f"    (no masking needed — cache only contains past + current tokens!)")

    weights = torch.softmax(scores, dim=-1)  # (1, step+1)
    print(f"\n  Attention weights (softmax):")
    print(f"    {cached_toks}:  {[round(x, 2) for x in weights[0].tolist()]}")

    # 4. Output: weights (1, step+1)  @  V_cache (step+1, 3)  →  (1, 3)
    out = weights @ V_cache    # (1, 3)
    cached_outputs.append(out[0])

    print(f"\n  Output = weights @ V_cache:  {[round(x, 2) for x in out[0].tolist()]}")

    # Show computation savings
    print(f"\n  💡 Projections computed this step:  1 token  (just '{tok}')")
    print(f"     K,V reused from cache:           {step} token(s)  {tokens[:step] if step > 0 else '(none yet)'}")


# ════════════════════════════════════════════════════════════════
# PART 3:  VERIFY — cached outputs match full recompute
# ════════════════════════════════════════════════════════════════
print(f"\n\n{'█' * 60}")
print(f"  PART 3:  VERIFY — do both methods agree?")
print(f"{'█' * 60}")

cached_out_tensor = torch.stack(cached_outputs)  # (T, d)

print_matrix("Full recompute output", out_full, tokens)
print_matrix("KV-cached output", cached_out_tensor, tokens)

match = torch.allclose(out_full, cached_out_tensor, atol=1e-5)
print(f"\n  ✅ Outputs match: {match}")

print(f"""
  ═══════════════════════════════════════════════════════
  SUMMARY — WHY KV CACHING MATTERS
  ═══════════════════════════════════════════════════════

  Generating the 5th token (E):

    Without cache:  Recompute K,V for A,B,C,D,E = 5 projections
    With cache:     Compute K,V for E only      = 1 projection
                    Reuse A,B,C,D from cache     = FREE

  For a sequence of length L:
    Without cache:  1 + 2 + 3 + ... + L = O(L²) projections total
    With cache:     1 + 1 + 1 + ... + 1 = O(L)  projections total

  The cache trades MEMORY (storing past K,V)
  for COMPUTE (not re-projecting them every step).
  ═══════════════════════════════════════════════════════
""")


████████████████████████████████████████████████████████████
  PART 1:  FULL RECOMPUTE  (no cache — training mode)
████████████████████████████████████████████████████████████

  All T tokens are processed at once.
  We compute Q, K, V for EVERY token,
  then do full (T × T) causal-masked attention.


  X  (input embeddings)  (shape [5, 3])
         ['d0', 'd1', 'd2']
    A    [1.0, 0.0, 2.0]
    B    [0.0, 1.0, 1.0]
    C    [2.0, 1.0, 0.0]
    D    [1.0, 2.0, 1.0]
    E    [0.0, 0.0, 2.0]

  Q = X @ Wq  (shape [5, 3])
         ['d0', 'd1', 'd2']
    A    [1.0, 0.0, 2.0]
    B    [0.0, 1.0, 1.0]
    C    [2.0, 1.0, 0.0]
    D    [1.0, 2.0, 1.0]
    E    [0.0, 0.0, 2.0]

  K = X @ Wk  (shape [5, 3])
         ['d0', 'd1', 'd2']
    A    [0.0, 1.0, 2.0]
    B    [1.0, 0.0, 1.0]
    C    [1.0, 2.0, 0.0]
    D    [2.0, 1.0, 1.0]
    E    [0.0, 0.0, 2.0]

  V = X @ Wv  (shape [5, 3])
         ['d0', 'd1', 'd2']
    A    [1.0, 2.0, 0.0]
    B    [0.0, 1.0, 1.0]
    C    [2.0, 0.0, 1.0]
    D

In [7]:
import torch
torch.set_printoptions(precision=2)

# ── Config ──────────────────────────────────────────────────────
T = 5
d = 3
tokens = ["A", "B", "C", "D", "E"]

X = torch.tensor([
    [1, 0, 2],   # A
    [0, 1, 1],   # B
    [2, 1, 0],   # C
    [1, 2, 1],   # D
    [0, 0, 2],   # E
], dtype=torch.float)

Wq = torch.eye(d)
Wk = torch.tensor([[0,1,0],[1,0,0],[0,0,1]], dtype=torch.float)
Wv = torch.tensor([[1,0,0],[0,0,1],[0,1,0]], dtype=torch.float)

scale = d ** 0.5

def print_matrix(label, mat, row_labels=None, col_labels=None):
    print(f"\n  {label}  (shape {list(mat.shape)})")
    if row_labels is None:
        row_labels = [str(i) for i in range(mat.shape[0])]
    if col_labels is None:
        col_labels = [f"d{i}" for i in range(mat.shape[-1])]
    print(f"    {'':4} {col_labels}")
    for i, lbl in enumerate(row_labels):
        print(f"    {lbl:4} {[round(x, 2) for x in mat[i].tolist()]}")


print("█" * 60)
print("  AUTOREGRESSIVE GENERATION — NO KV CACHE")
print("█" * 60)
print("""
  At each step we want the output for the LATEST token only.
  But without a cache, we must:
    1. Feed ALL tokens seen so far
    2. Recompute Q, K, V for ALL of them
    3. Build and mask the FULL score matrix
    4. Only keep the LAST ROW of the output

  Everything above the last row is WASTED WORK.
""")

total_projections = 0
outputs = []

for step in range(T):
    toks_so_far = tokens[:step+1]
    X_so_far = X[:step+1]       # (step+1, d)
    n = step + 1

    print(f"\n{'═' * 60}")
    print(f"  STEP {step}:  Generating after [{', '.join(toks_so_far)}]")
    print(f"  We need the output for '{toks_so_far[-1]}' only.")
    print(f"{'═' * 60}")

    # ── Recompute Q, K, V for ALL tokens so far ────────────────
    Q = X_so_far @ Wq     # (n, d)
    K = X_so_far @ Wk     # (n, d)
    V = X_so_far @ Wv     # (n, d)

    total_projections += n   # counting how many tokens we projected

    print_matrix("Input X  (ALL tokens so far)", X_so_far, toks_so_far)
    print_matrix("Q = X @ Wq  (recomputed for ALL)", Q, toks_so_far)
    print_matrix("K = X @ Wk  (recomputed for ALL)", K, toks_so_far)
    print_matrix("V = X @ Wv  (recomputed for ALL)", V, toks_so_far)

    print(f"\n  ⚠️  Projected {n} token(s) but we only need '{toks_so_far[-1]}'s output!")
    if step > 0:
        print(f"      K,V for [{', '.join(toks_so_far[:-1])}] were already computed last step — wasted!")

    # ── Full score matrix ───────────────────────────────────────
    scores = (Q @ K.T) / scale   # (n, n)

    # Causal mask
    causal_mask = torch.triu(torch.full((n, n), float('-inf')), diagonal=1)

    print_matrix("Raw scores  (Q @ Kᵀ / √d)", scores, toks_so_far, toks_so_far)

    print(f"\n  Causal mask  ({n}×{n}):")
    print(f"    {'':4} {toks_so_far}")
    for i, tok in enumerate(toks_so_far):
        row = ["  ✓" if causal_mask[i,j] == 0 else "  ✗" for j in range(n)]
        print(f"    {tok}  {''.join(row)}")

    scores_masked = scores + causal_mask
    weights = torch.softmax(scores_masked, dim=-1)    # (n, n)
    out_all = weights @ V                              # (n, d)

    print_matrix("Attention weights  (full matrix)", weights, toks_so_far, toks_so_far)
    print_matrix("Output  (ALL rows — full matrix)", out_all, toks_so_far)

    # ── We only use the LAST row ────────────────────────────────
    out_last = out_all[-1:]   # (1, d)
    outputs.append(out_last[0])

    print(f"\n  ✂️  We ONLY keep the last row ('{toks_so_far[-1]}'):")
    print(f"      output = {[round(x,2) for x in out_last[0].tolist()]}")
    if step > 0:
        print(f"\n      Rows for [{', '.join(toks_so_far[:-1])}] were computed and THROWN AWAY.")


# ── Summary ─────────────────────────────────────────────────────
print(f"\n\n{'█' * 60}")
print(f"  WASTE SUMMARY")
print(f"{'█' * 60}")

print(f"\n  Step-by-step projection count:")
for step in range(T):
    n = step + 1
    toks = tokens[:n]
    wasted = n - 1
    print(f"    Step {step} ('{tokens[step]}'): projected {n} tokens, "
          f"needed 1, wasted {wasted}  [{', '.join(toks)}]")

print(f"""
  Total K,V projections:  1 + 2 + 3 + 4 + 5 = {total_projections}
  Actually needed:        1 + 1 + 1 + 1 + 1 = {T}
  Wasted:                 {total_projections - T} projections  ({(total_projections - T)/total_projections*100:.0f}% waste)

  This is O(L²) projections to generate L tokens.
  With KV cache it's O(L) — store old K,V, only project the new token.

  (And the score matrix itself grows too:
   1×1 + 2×2 + 3×3 + 4×4 + 5×5 = {sum(i*i for i in range(1,T+1))} score entries
   vs. 1 + 2 + 3 + 4 + 5 = {total_projections} with cache — just one row each step)
""")

# ── Verify outputs match ────────────────────────────────────────
stacked = torch.stack(outputs)
print_matrix("Final collected outputs (one per step)", stacked, tokens)


████████████████████████████████████████████████████████████
  AUTOREGRESSIVE GENERATION — NO KV CACHE
████████████████████████████████████████████████████████████

  At each step we want the output for the LATEST token only.
  But without a cache, we must:
    1. Feed ALL tokens seen so far
    2. Recompute Q, K, V for ALL of them
    3. Build and mask the FULL score matrix
    4. Only keep the LAST ROW of the output

  Everything above the last row is WASTED WORK.


════════════════════════════════════════════════════════════
  STEP 0:  Generating after [A]
  We need the output for 'A' only.
════════════════════════════════════════════════════════════

  Input X  (ALL tokens so far)  (shape [1, 3])
         ['d0', 'd1', 'd2']
    A    [1.0, 0.0, 2.0]

  Q = X @ Wq  (recomputed for ALL)  (shape [1, 3])
         ['d0', 'd1', 'd2']
    A    [1.0, 0.0, 2.0]

  K = X @ Wk  (recomputed for ALL)  (shape [1, 3])
         ['d0', 'd1', 'd2']
    A    [0.0, 1.0, 2.0]

  V = X @ Wv  (recomputed 